In [0]:
# NB_hospital_daily_summary — Gold Layer: Hospital Daily Summary
# STTM: One row per hospital per calendar day
# Load Strategy: FULL — CREATE OR REPLACE TABLE ... AS SELECT
# Sources: silver.lab_results (driving), silver.hospitals, silver.dim_date
# Window: LAG() partitioned by hospital_id, ordered by summary_date

from pyspark.sql import functions as F
from pyspark.sql import Window
from pyspark.sql.types import *
import uuid
from datetime import datetime

# --- Configuration ---
CATALOG = "caresync_databricks_workspace"
SOURCE_SCHEMA = "silver"
TARGET_SCHEMA = "gold"
TARGET_TABLE = f"{CATALOG}.{TARGET_SCHEMA}.hospital_daily_summary"

RUN_ID = str(uuid.uuid4())
RUN_START = datetime.utcnow()

print("=" * 60)
print("NB_hospital_daily_summary — Gold Layer Pipeline")
print("=" * 60)
print(f"  Run ID             : {RUN_ID}")
print(f"  Run Start (UTC)    : {RUN_START}")
print(f"  Target Table       : {TARGET_TABLE}")
print("=" * 60)

In [0]:
print("=" * 60)
print("STEP 1: Read Source Tables")
print("=" * 60)

lab_results_df = spark.table(f"{CATALOG}.{SOURCE_SCHEMA}.lab_results")
hospitals_df = spark.table(f"{CATALOG}.{SOURCE_SCHEMA}.hospitals")
dim_date_df = spark.table(f"{CATALOG}.{SOURCE_SCHEMA}.dim_date")

lab_results_count = lab_results_df.count()
hospitals_count = hospitals_df.count()
dim_date_count = dim_date_df.count()

print(f"  lab_results   : {lab_results_count:,} rows")
print(f"  hospitals     : {hospitals_count:,} rows")
print(f"  dim_date      : {dim_date_count:,} rows")
print("=" * 60)

In [0]:
# Driving table: lab_results grouped by (hospital_id, test_date)
# This establishes the grain: one row per hospital per day that has activity.

print("=" * 60)
print("STEP 2: Aggregate Lab Results by Hospital + Date")
print("=" * 60)

daily_agg_df = (
    lab_results_df
    .groupBy("hospital_id", "test_date")
    .agg(
        F.count("*").alias("total_tests"),
        F.countDistinct("patient_id").alias("unique_patients_tested"),
        F.countDistinct("doctor_id").alias("active_doctor_count"),
        F.count(F.when(F.col("result_status") == "Abnormal", 1)).alias("abnormal_test_count"),
        F.count(F.when(F.col("result_status") == "Critical", 1)).alias("critical_test_count")
    )
    .withColumnRenamed("test_date", "summary_date")
)

agg_count = daily_agg_df.count()
print(f"  Aggregated rows (hospital x date) : {agg_count:,}")
print("=" * 60)

In [0]:
# LEFT JOIN hospitals for hospital_name
# LEFT JOIN dim_date for day_name, is_weekend (join on date_key = CAST(date_format(summary_date, 'yyyyMMdd') AS INT))
# Compute rates with safe division (NULL when denominator = 0)
# Compute LAG(total_tests) for day-over-day trend

print("=" * 60)
print("STEP 3: Join Hospitals and dim_date, Compute Derived Columns")
print("=" * 60)

# Add date_key for joining to dim_date
daily_with_key_df = daily_agg_df.withColumn(
    "_date_key", F.date_format(F.col("summary_date"), "yyyyMMdd").cast("int")
)

# Join hospitals and dim_date
joined_df = (
    daily_with_key_df.alias("d")
    .join(
        hospitals_df.alias("h"),
        F.col("d.hospital_id") == F.col("h.hospital_id"),
        "left"
    )
    .join(
        dim_date_df.alias("cal"),
        F.col("d._date_key") == F.col("cal.date_key"),
        "left"
    )
)

# Compute derived rate columns with safe division
with_rates_df = (
    joined_df
    .withColumn(
        "abnormal_rate_pct",
        F.when(
            F.col("d.total_tests") == 0, F.lit(None)
        ).otherwise(
            F.round(F.col("d.abnormal_test_count") / F.col("d.total_tests") * 100, 2)
        ).cast("decimal(5,2)")
    )
    .withColumn(
        "critical_rate_pct",
        F.when(
            F.col("d.total_tests") == 0, F.lit(None)
        ).otherwise(
            F.round(F.col("d.critical_test_count") / F.col("d.total_tests") * 100, 2)
        ).cast("decimal(5,2)")
    )
)

# Window for LAG: partition by hospital_id, order by summary_date
# LAG compares to the previous available observation (not calendar day)
w = Window.partitionBy("d.hospital_id").orderBy("d.summary_date")

with_lag_df = (
    with_rates_df
    .withColumn("prev_day_total_tests", F.lag("d.total_tests").over(w).cast("int"))
    .withColumn(
        "test_volume_change_pct",
        F.when(
            (F.col("prev_day_total_tests").isNull()) | (F.col("prev_day_total_tests") == 0),
            F.lit(None)
        ).otherwise(
            F.round(
                (F.col("d.total_tests") - F.col("prev_day_total_tests")) / F.col("prev_day_total_tests") * 100, 2
            )
        ).cast("decimal(6,2)")
    )
)

# Select final columns in STTM order
hospital_daily_df = with_lag_df.select(
    F.col("d.hospital_id").cast("int").alias("hospital_id"),
    F.col("h.hospital_name").cast("string").alias("hospital_name"),
    F.col("d.summary_date").cast("date").alias("summary_date"),
    F.col("cal.day_name").cast("string").alias("day_name"),
    F.col("cal.is_weekend").cast("boolean").alias("is_weekend"),
    F.col("d.total_tests").cast("int").alias("total_tests"),
    F.col("unique_patients_tested").cast("int").alias("unique_patients_tested"),
    F.col("active_doctor_count").cast("int").alias("active_doctor_count"),
    F.col("d.abnormal_test_count").cast("int").alias("abnormal_test_count"),
    F.col("d.critical_test_count").cast("int").alias("critical_test_count"),
    F.col("abnormal_rate_pct"),
    F.col("critical_rate_pct"),
    F.col("prev_day_total_tests"),
    F.col("test_volume_change_pct")
)

print(f"  Transformation built successfully.")
print(f"  Output columns: {hospital_daily_df.columns}")
print("=" * 60)

In [0]:
print("=" * 60)
print("STEP 4: Pre-write Validation")
print("=" * 60)

hospital_daily_df.createOrReplaceTempView("hospital_daily_staging")
staging_count = spark.sql("SELECT COUNT(*) AS cnt FROM hospital_daily_staging").collect()[0]["cnt"]

print(f"  Staging row count: {staging_count:,}")

# Validation 1: Grain uniqueness (hospital_id + summary_date)
grain_dupes = spark.sql("""
    SELECT hospital_id, summary_date, COUNT(*) AS cnt
    FROM hospital_daily_staging
    GROUP BY hospital_id, summary_date
    HAVING cnt > 1
""").count()
assert grain_dupes == 0, f"FAIL: {grain_dupes} duplicate (hospital_id, summary_date) found."
print(f"  [PASS] Grain unique: (hospital_id, summary_date) — 0 duplicates")

# Validation 2: No null grain keys
null_grain = spark.sql("""
    SELECT COUNT(*) AS cnt FROM hospital_daily_staging 
    WHERE hospital_id IS NULL OR summary_date IS NULL
""").collect()[0]["cnt"]
assert null_grain == 0, f"FAIL: {null_grain} rows with null grain keys."
print(f"  [PASS] No null grain keys")

# Validation 3: total_tests > 0 (every row is driven by lab_results, so must be >= 1)
zero_tests = spark.sql("SELECT COUNT(*) AS cnt FROM hospital_daily_staging WHERE total_tests <= 0").collect()[0]["cnt"]
assert zero_tests == 0, f"FAIL: {zero_tests} rows with total_tests <= 0."
print(f"  [PASS] All total_tests >= 1")

# Validation 4: Rates in valid range [0, 100]
bad_rates = spark.sql("""
    SELECT COUNT(*) AS cnt FROM hospital_daily_staging 
    WHERE (abnormal_rate_pct < 0 OR abnormal_rate_pct > 100)
       OR (critical_rate_pct < 0 OR critical_rate_pct > 100)
""").collect()[0]["cnt"]
assert bad_rates == 0, f"FAIL: {bad_rates} rows with rates outside [0,100]."
print(f"  [PASS] All rates in [0, 100] range")

# Validation 5: Source row count reconciliation
# Sum of total_tests across all rows should equal lab_results_count
total_tests_sum = spark.sql("SELECT SUM(total_tests) AS s FROM hospital_daily_staging").collect()[0]["s"]
assert total_tests_sum == lab_results_count, (
    f"FAIL: SUM(total_tests) {total_tests_sum} != lab_results count {lab_results_count}"
)
print(f"  [PASS] SUM(total_tests) = {total_tests_sum:,} matches source lab_results ({lab_results_count:,})")

# Validation 6: Schema check
expected_cols = [
    "hospital_id", "hospital_name", "summary_date", "day_name", "is_weekend",
    "total_tests", "unique_patients_tested", "active_doctor_count",
    "abnormal_test_count", "critical_test_count", "abnormal_rate_pct",
    "critical_rate_pct", "prev_day_total_tests", "test_volume_change_pct"
]
actual_cols = [f.name for f in spark.sql("SELECT * FROM hospital_daily_staging LIMIT 0").schema.fields]
assert actual_cols == expected_cols, f"FAIL: Column mismatch.\nExpected: {expected_cols}\nActual:   {actual_cols}"
print(f"  [PASS] Schema matches STTM specification ({len(expected_cols)} columns)")

print("-" * 60)
print(f"  All pre-write validations PASSED.")
print("=" * 60)

In [0]:
print("=" * 60)
print(f"STEP 5: Write to Gold — {TARGET_TABLE}")
print("=" * 60)
print(f"  Strategy: CREATE OR REPLACE TABLE ... AS SELECT")

spark.sql(f"""
    CREATE OR REPLACE TABLE {TARGET_TABLE} AS
    SELECT * FROM hospital_daily_staging
""")

print(f"  ✅ Table created/replaced successfully.")
print("=" * 60)

In [0]:
print("=" * 60)
print("STEP 6: Post-write Verification")
print("=" * 60)

gold_df = spark.table(TARGET_TABLE)
gold_count = gold_df.count()

print(f"  Gold table row count   : {gold_count:,}")
print(f"  Expected (from staging): {staging_count:,}")
assert gold_count == staging_count, f"FAIL: Gold count {gold_count} != staging count {staging_count}"
print(f"  [PASS] Row counts match.")

print("-" * 60)
print("  Schema:")
gold_df.printSchema()
print("-" * 60)
print("  Sample rows (5):")
gold_df.show(5, truncate=False)
print("=" * 60)

In [0]:
from datetime import datetime
import json

RUN_END = datetime.utcnow()
duration_sec = (RUN_END - RUN_START).total_seconds()

exit_payload = {
    "status": "SUCCESS",
    "run_id": RUN_ID,
    "target_table": TARGET_TABLE,
    "rows_written": gold_count,
    "source_lab_results_count": lab_results_count,
    "unique_hospital_day_combos": staging_count,
    "run_start_utc": str(RUN_START),
    "run_end_utc": str(RUN_END),
    "duration_seconds": round(duration_sec, 2)
}

print("=" * 60)
print("STEP 7: Notebook Exit")
print("=" * 60)
for k, v in exit_payload.items():
    print(f"  {k:30s}: {v}")
print("=" * 60)

In [0]:
record_count = gold_count

print("=" * 60)
print("STEP 8: Simple Record Count Exit")
print("=" * 60)
print(f"  Records processed  : {record_count:,}")
print(f"  Returning value    : {record_count}")
print("=" * 60)

dbutils.notebook.exit(str(record_count))